In [6]:
import polars as pl
import gc
from datetime import datetime
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, classification_report
from features import apply_type_casting, generate_features, cat_features

In [ ]:
df = pl.scan_parquet('../data/train_features_v4.parquet')

df_train_lazy = df.filter(pl.col('event_dttm') < datetime(2025, 5, 1))

df_train_fraud = df_train_lazy.filter((pl.col('target') == 1)).collect()
df_train_nonfraud = df_train_lazy.filter((pl.col('target') == 0)).collect().sample(n=2_000_000, seed=42)

df_train = pl.concat([df_train_fraud, df_train_nonfraud]).sample(fraction=1.0, shuffle=True, seed=42)

del df_train_fraud, df_train_nonfraud, df_train_lazy
gc.collect()

df_val = df.filter(
    (pl.col('event_date').dt.year() == 2025) & 
    (pl.col('event_date').dt.month() == 5)
).collect()

Размеры:
X_train: (2044441, 38)
X_val: (12575998, 38)

Количество фрода:
y_train: 44441
y_val: 6997


In [ ]:
drop_cols =['customer_id', 'event_id', 'event_dttm', 'event_date', 'target']

X_train = df_train.drop(drop_cols).to_pandas()
y_train = df_train.select('target').to_pandas()

X_val = df_val.drop(drop_cols).to_pandas()
y_val = df_val.select('target').to_pandas()

del df_train, df_val
gc.collect()

In [ ]:
model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.05,
    depth=5,
    l2_leaf_reg=5.0,
    task_type='GPU',
    scale_pos_weight=45,
    eval_metric='PRAUC',
    metric_period=50,
    od_type='Iter',
    od_wait=50,
    random_seed=42
)

model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_val, y_val), verbose=50)
preds = model.predict_proba(X_val)[:, 1]
metric = average_precision_score(y_val, preds)

print(metric)

Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.7889586	test: 0.1428522	best: 0.1428522 (0)	total: 845ms	remaining: 11m 15s
50:	learn: 0.8551877	test: 0.2578555	best: 0.2578555 (50)	total: 46.8s	remaining: 11m 27s
100:	learn: 0.8678418	test: 0.2915334	best: 0.2915334 (100)	total: 1m 57s	remaining: 13m 36s
150:	learn: 0.8734977	test: 0.3052041	best: 0.3052041 (150)	total: 3m 13s	remaining: 13m 53s
200:	learn: 0.8774681	test: 0.3134504	best: 0.3134504 (200)	total: 4m 31s	remaining: 13m 28s
250:	learn: 0.8803677	test: 0.3208043	best: 0.3208043 (250)	total: 5m 48s	remaining: 12m 41s
300:	learn: 0.8825660	test: 0.3239254	best: 0.3239254 (300)	total: 7m 5s	remaining: 11m 45s
350:	learn: 0.8846281	test: 0.3285001	best: 0.3285001 (350)	total: 8m 23s	remaining: 10m 43s
400:	learn: 0.8861631	test: 0.3311995	best: 0.3311995 (400)	total: 9m 39s	remaining: 9m 37s
450:	learn: 0.8875684	test: 0.3335103	best: 0.3335354 (449)	total: 10m 57s	remaining: 8m 28s
500:	learn: 0.8888466	test: 0.3360250	best: 0.3360250 (500)	total: 12m 16s	remai

In [8]:
model.get_feature_importance(prettified=True)

,Feature Id,Importances
0,mcc_code,11.287603
1,client_op_seq_num,8.640962
2,amt_sum_1h,8.413907
3,amt_sum_24h,8.413797
4,amt_diff_from_exp_mean,6.714003
5,op_count_1h,6.327110
6,client_expanding_mean_amt,5.935250
7,event_desc,5.886532
8,op_count_24h,5.018895
9,prev_event_desc,3.850639


In [9]:
model.save_model('../models/catboost_v4.cbm')